# Lab 7: Solving Backwards

## Linear systems as reverse engineering

In Chapter 7, a matrix is a machine and a linear system is a request to run that machine backwards.

The central equation is

$$
Ax=b.
$$

In this lab, you will use Python to explore four connected views:

1. **Equation view:** solve several equations at once.
2. **Geometry view:** find intersections of lines.
3. **Column view:** build a target vector from columns.
4. **Computation view:** use numerical tools and check residuals.

This is not just a quick practice. It is a guided computational story.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## 1. Forward first, then backward

Let

$$
A=\begin{bmatrix}2&1\\1&3\end{bmatrix},
\qquad
x=\begin{bmatrix}2\\3\end{bmatrix}.
$$

The forward problem computes $b=Ax$. The backward problem tries to recover $x$ from $A$ and $b$.

In [ ]:
A = np.array([[2, 1],
              [1, 3]], dtype=float)

x_true = np.array([2, 3], dtype=float)
b = A @ x_true

print("A =
", A)
print("x_true =", x_true)
print("b = A @ x_true =", b)

In [ ]:
x_recovered = np.linalg.solve(A, b)
print("x_recovered =", x_recovered)
print("A @ x_recovered =", A @ x_recovered)
print("recovery error =", np.linalg.norm(x_recovered - x_true))

### Student task

Change `x_true` to another vector. Recompute `b`, then solve the system. Does Python recover your original input?

## 2. Visualizing two equations as two lines

The system

$$
2x_1+x_2=7,
\qquad
x_1+3x_2=11
$$

has one solution because the two lines intersect once.

In [ ]:
def plot_2x2_system(A, b, xlim=(-2, 8), ylim=(-2, 8), title="Linear system"):
    xs = np.linspace(xlim[0], xlim[1], 500)
    plt.figure(figsize=(7, 6))
    for i in range(2):
        a, c = A[i, 0], A[i, 1]
        if abs(c) > 1e-12:
            ys = (b[i] - a * xs) / c
            plt.plot(xs, ys, label=f"{a:.2g}x1 + {c:.2g}x2 = {b[i]:.2g}")
        elif abs(a) > 1e-12:
            plt.axvline(b[i] / a, label=f"{a:.2g}x1 = {b[i]:.2g}")
    try:
        sol = np.linalg.solve(A, b)
        plt.scatter([sol[0]], [sol[1]], s=100, zorder=5, label=f"solution {sol.round(3)}")
    except np.linalg.LinAlgError:
        pass
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.grid(True, alpha=0.35)
    plt.legend()
    plt.show()

plot_2x2_system(A, b, title="One solution: two lines meet")

## 3. Three cases: one, none, infinitely many

The geometry of two equations in two unknowns has three possibilities.

In [ ]:
examples = {
    "one solution": (np.array([[2, 1], [1, 3]], dtype=float), np.array([7, 11], dtype=float)),
    "no solution": (np.array([[1, 1], [1, 1]], dtype=float), np.array([2, 5], dtype=float)),
    "infinitely many": (np.array([[1, 1], [2, 2]], dtype=float), np.array([2, 4], dtype=float)),
}

for name, (Ai, bi) in examples.items():
    print("
---", name, "---")
    print("rank(A) =", np.linalg.matrix_rank(Ai))
    print("rank([A|b]) =", np.linalg.matrix_rank(np.column_stack([Ai, bi])))
    try:
        print("solve:", np.linalg.solve(Ai, bi))
    except np.linalg.LinAlgError as e:
        print("np.linalg.solve failed:", e)

In [ ]:
plot_2x2_system(examples["one solution"][0], examples["one solution"][1], title="One solution")
plot_2x2_system(examples["no solution"][0], examples["no solution"][1], title="No solution: parallel lines", xlim=(-2,6), ylim=(-2,6))
plot_2x2_system(examples["infinitely many"][0], examples["infinitely many"][1], title="Infinitely many: same line", xlim=(-2,5), ylim=(-2,5))

### Student task

Create your own examples of the three cases. Use `np.linalg.matrix_rank` to diagnose them.

## 4. Column picture: the solution is a recipe

For

$$
A=\begin{bmatrix}2&1\\1&3\end{bmatrix},
$$

the columns are

$$
a_1=\begin{bmatrix}2\\1\end{bmatrix},
\qquad
a_2=\begin{bmatrix}1\\3\end{bmatrix}.
$$

Solving $Ax=b$ means finding coefficients $x_1,x_2$ such that

$$
x_1a_1+x_2a_2=b.
$$

In [ ]:
a1 = A[:, 0]
a2 = A[:, 1]
x = np.linalg.solve(A, b)

print("a1 =", a1)
print("a2 =", a2)
print("coefficients x =", x)
print("x[0]*a1 + x[1]*a2 =", x[0]*a1 + x[1]*a2)
print("b =", b)

In [ ]:
def draw_column_recipe(a1, a2, x, b):
    plt.figure(figsize=(7, 7))
    origin = np.array([0, 0])
    v1 = x[0] * a1
    v2 = x[1] * a2
    plt.quiver(*origin, *v1, angles='xy', scale_units='xy', scale=1, width=0.008, label='x1 a1')
    plt.quiver(*v1, *v2, angles='xy', scale_units='xy', scale=1, width=0.008, label='x2 a2')
    plt.quiver(0, 0, b[0], b[1], angles='xy', scale_units='xy', scale=1, width=0.004, alpha=0.8, label='b')
    pts = np.vstack([origin, v1, b])
    pad = 2
    plt.xlim(min(pts[:,0])-pad, max(pts[:,0])+pad)
    plt.ylim(min(pts[:,1])-pad, max(pts[:,1])+pad)
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.grid(True, alpha=0.35)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.legend()
    plt.title('Column recipe: build b from columns')
    plt.show()

draw_column_recipe(a1, a2, x, b)

## 5. Implement a simple elimination step

To understand solving, we can implement row operations ourselves.

In [ ]:
M = np.array([[2, 1, 7],
              [1, 3, 11]], dtype=float)

print("Original augmented matrix:")
print(M)

# Swap rows so the first pivot is 1
M[[0, 1]] = M[[1, 0]]
print("
After swapping rows:")
print(M)

# Eliminate below pivot
M[1] = M[1] - 2*M[0]
print("
After R2 <- R2 - 2 R1:")
print(M)

# Scale second row
M[1] = M[1] / M[1, 1]
print("
After scaling R2:")
print(M)

# Eliminate above pivot
M[0] = M[0] - 3*M[1]
print("
Reduced form:")
print(M)

### Student task

Repeat the row-reduction process for

$$
\begin{bmatrix}
3 & 2 & 12\\
1 & -1 & 1
\end{bmatrix}.
$$

## 6. Residuals: checking an answer

For a proposed solution $x$, the residual is

$$
r=b-Ax.
$$

If $r=0$, then $x$ solves the system exactly. If $r$ is small, then $x$ is close to solving the system.

In [ ]:
A = np.array([[2, 1], [1, 3]], dtype=float)
b = np.array([7, 11], dtype=float)

candidates = [
    np.array([2, 3], dtype=float),
    np.array([2.1, 2.9], dtype=float),
    np.array([0, 0], dtype=float),
]

for x in candidates:
    r = b - A @ x
    print("x =", x, " residual =", r, " residual norm =", np.linalg.norm(r))

## 7. Nearly singular systems

Some systems technically have a unique solution but are numerically sensitive. A tiny change in $b$ can cause a large change in $x$.

This is a first glimpse of conditioning.

In [ ]:
A = np.array([[1, 1],
              [1, 1.001]], dtype=float)

b1 = np.array([2, 2.001], dtype=float)
b2 = np.array([2, 2.002], dtype=float)

x1 = np.linalg.solve(A, b1)
x2 = np.linalg.solve(A, b2)

print("x for b1:", x1)
print("x for b2:", x2)
print("change in b:", np.linalg.norm(b2 - b1))
print("change in x:", np.linalg.norm(x2 - x1))
print("condition number:", np.linalg.cond(A))

### Reflection

Why is this dangerous in data problems? What if measurements contain noise?

## 8. Application: recovering prices from bundles

In [ ]:
# Rows are purchases; columns are item counts.
# Unknowns are prices.
A = np.array([[2, 1, 0],
              [1, 0, 3],
              [0, 2, 2]], dtype=float)

true_prices = np.array([1.50, 0.80, 2.00])
totals = A @ true_prices

recovered_prices = np.linalg.solve(A, totals)

print("purchase matrix:
", A)
print("totals:", totals)
print("recovered prices:", recovered_prices)

### Student task

Add a fourth item and create a $4\times 4$ purchase matrix. Choose true prices, compute totals, and recover the prices.

## 9. Application: unmixing signals

Two hidden signals are mixed by a matrix. If the matrix is invertible, we can recover the hidden signals.

In [ ]:
t = np.linspace(0, 2*np.pi, 500)
s1 = np.sin(t)
s2 = np.sign(np.sin(3*t))
S = np.vstack([s1, s2])

Mix = np.array([[0.8, 0.3],
                [0.2, 0.9]])
Y = Mix @ S
S_hat = np.linalg.solve(Mix, Y)

plt.figure(figsize=(10, 4))
plt.plot(t, S[0], label='original signal 1')
plt.plot(t, S[1], label='original signal 2')
plt.title('Original hidden signals')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(t, Y[0], label='mixed measurement 1')
plt.plot(t, Y[1], label='mixed measurement 2')
plt.title('Observed mixtures')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(t, S_hat[0], label='recovered signal 1')
plt.plot(t, S_hat[1], label='recovered signal 2')
plt.title('Recovered signals by solving backwards')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("recovery error:", np.linalg.norm(S_hat - S))

## 10. High-dimensional recovery

Now solve a system with many variables. The story is the same: generate an input, send it forward, then recover it.

In [ ]:
np.random.seed(42)

n = 300
A = np.random.randn(n, n)
x_true = np.random.randn(n)
b = A @ x_true

x_hat = np.linalg.solve(A, b)

relative_error = np.linalg.norm(x_hat - x_true) / np.linalg.norm(x_true)
relative_residual = np.linalg.norm(A @ x_hat - b) / np.linalg.norm(b)

print("n =", n)
print("relative recovery error:", relative_error)
print("relative residual:", relative_residual)
print("condition number:", np.linalg.cond(A))

## 11. Final challenge

Design your own reverse-engineering problem.

Your problem should include:

1. A real-world story.
2. A matrix $A$.
3. A hidden vector $x$.
4. An observed vector $b=Ax$.
5. A solution using Python.
6. A written interpretation of the recovered vector.

Possible themes:

- prices from purchase totals,
- ingredients from nutrition measurements,
- hidden signals from sensor readings,
- traffic flows from road counts,
- weights in a simple prediction rule.